# WNBA RAPM + 4-Factor RAPM

Self-contained notebook — no local imports needed.

**What you need:**
- The `wnba_data/` folder in the same directory as this notebook
- `stints/stints_{year}_RS.csv` for standard RAPM
- `stints_rich/stints_rich_{year}_RS.csv` for 4F factor rates
- `player_names.csv`

**Models:**
1. **Standard RAPM** — ridge regression: +1 offense / +1 defense design matrix, y = points scored
2. **4F RAPM** — decomposes RAPM into 8 on-court factor rates via OLS (TS%, TOV%, OREB%, Trans%)

In [ ]:
%pip install -q numpy pandas scipy scikit-learn

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.linear_model import Ridge

# ── CONFIG ────────────────────────────────────────────────────────────────────
# Set DATA_DIR to wherever wnba_data/ lives.
# Default: same folder as this notebook.
DATA_DIR = Path("wnba_data")

# Seasons to pool for RAPM (all are weighted equally — add earlier years for stability)
RAPM_SEASONS  = [2023, 2024, 2025]   # adjust as needed

# Ridge penalty (higher = more shrinkage toward 0)
LAMBDA = 2000

# Minimum total possessions to appear in output
MIN_POSS = 200

print(f"DATA_DIR   : {DATA_DIR.resolve()}")
print(f"RAPM years : {RAPM_SEASONS}")

## 1 · Load stints data

In [ ]:
def _load_stints(seasons: list[int], rich: bool = False) -> pd.DataFrame:
    """Load and concatenate stints CSVs for the requested seasons."""
    frames = []
    subdir = "stints_rich" if rich else "stints"
    prefix = "stints_rich" if rich else "stints"
    for yr in seasons:
        path = DATA_DIR / subdir / f"{prefix}_{yr}_RS.csv"
        if not path.exists():
            print(f"  ⚠  missing: {path}")
            continue
        df = pd.read_csv(path)
        df["season"] = yr
        frames.append(df)
        print(f"  loaded {yr}: {len(df):,} possessions")
    if not frames:
        raise FileNotFoundError(f"No stints files found in {DATA_DIR / subdir}")
    return pd.concat(frames, ignore_index=True)


print("Loading standard stints...")
stints = _load_stints(RAPM_SEASONS, rich=False)

print(f"\nLoading stints_rich...")
stints_rich = _load_stints(RAPM_SEASONS, rich=True)

# Player names
names_path = DATA_DIR / "player_names.csv"
player_names: dict[int, str] = {}
if names_path.exists():
    _n = pd.read_csv(names_path)
    player_names = dict(zip(_n["player_id"].astype(int), _n["player_name"]))
    print(f"\nLoaded {len(player_names):,} player names")
else:
    print(f"  ⚠  player_names.csv not found — IDs will be used")

OFF_COLS = [f"off_p{i}" for i in range(1, 6)]
DEF_COLS = [f"def_p{i}" for i in range(1, 6)]

print(f"\nTotal possessions: {len(stints):,}")

## 2 · Standard RAPM

Design matrix: each possession is a row.  
Player on **offense** → +1 in their column.  
Player on **defense** → +1 in their column.  

Two separate ridge regressions:
- **oRAPM**: offensive players (+1), predict points scored
- **dRAPM**: defensive players (+1), predict points allowed → negate coefficients

In [ ]:
def fit_rapm(df: pd.DataFrame, lam: float = 2000) -> pd.DataFrame:
    """
    Ridge-regression RAPM from a stints DataFrame.
    Returns DataFrame: player_id, orapm, drapm, net_rapm, poss
    """
    off_cols = [f"off_p{i}" for i in range(1, 6)]
    def_cols = [f"def_p{i}" for i in range(1, 6)]

    # Collect all player IDs
    all_ids = sorted(
        set(df[off_cols].values.flatten().tolist()) |
        set(df[def_cols].values.flatten().tolist())
    )
    all_ids = [int(p) for p in all_ids if not np.isnan(p)]
    pid_to_col = {pid: i for i, pid in enumerate(all_ids)}
    n_players = len(all_ids)
    n_poss    = len(df)

    # Build sparse design matrices
    off_r, off_c, def_r, def_c = [], [], [], []
    for row_idx, row in enumerate(df.itertuples(index=False)):
        for col in off_cols:
            pid = int(getattr(row, col))
            if pid in pid_to_col:
                off_r.append(row_idx); off_c.append(pid_to_col[pid])
        for col in def_cols:
            pid = int(getattr(row, col))
            if pid in pid_to_col:
                def_r.append(row_idx); def_c.append(pid_to_col[pid])

    X_off = csr_matrix(([1.0]*len(off_r), (off_r, off_c)), shape=(n_poss, n_players))
    X_def = csr_matrix(([1.0]*len(def_r), (def_r, def_c)), shape=(n_poss, n_players))
    y     = df["points"].values.astype(float)

    ridge = Ridge(alpha=lam, fit_intercept=True)
    ridge.fit(X_off, y)
    orapm = ridge.coef_ * 100   # per-100-possession scale

    ridge.fit(X_def, y)
    drapm = -ridge.coef_ * 100  # negate: positive = good defense

    # Possession counts
    poss_map: dict[int, int] = {}
    for col in off_cols + def_cols:
        for pid, cnt in df[col].value_counts().items():
            pid = int(pid)
            poss_map[pid] = poss_map.get(pid, 0) + int(cnt)

    return pd.DataFrame({
        "player_id": all_ids,
        "orapm":     np.round(orapm, 3),
        "drapm":     np.round(drapm, 3),
        "net_rapm":  np.round(orapm + drapm, 3),
        "poss":      [poss_map.get(pid, 0) for pid in all_ids],
    })


print(f"Fitting RAPM on {len(stints):,} possessions...")
rapm = fit_rapm(stints, lam=LAMBDA)
rapm = rapm[rapm["poss"] >= MIN_POSS].copy()
rapm["player_name"] = rapm["player_id"].map(player_names).fillna(rapm["player_id"].astype(str))

print(f"Players qualifying (≥{MIN_POSS} poss): {len(rapm)}")
(
    rapm[["player_name","net_rapm","orapm","drapm","poss"]]
    .sort_values("net_rapm", ascending=False)
    .head(25)
    .reset_index(drop=True)
    .style.format({"net_rapm":"+.2f","orapm":"+.2f","drapm":"+.2f","poss":",d"})
    .background_gradient(subset=["net_rapm"], cmap="RdYlGn")
    .set_caption(f"Top 25 — RAPM ({', '.join(str(y) for y in RAPM_SEASONS)} pooled, λ={LAMBDA:,})")
)

## 3 · Per-player four-factor rates

Uses `stints_rich` which has per-possession stats: `fga, fgm, fg3a, fg3m, fta, ftm, tov_flag, oreb, oreb_chance, trans_flag`.

**Offensive factors** (aggregated over possessions where player is on offense):
| Column | Formula | Meaning |
|--------|---------|------|
| `raw_ots` | `(fgm·2 + fg3m + ftm) / (fga·2 + fg3a + fta·0.44)` | TS-like rate |
| `raw_otov` | `tov / (fga + 0.44·fta + tov)` | Turnover rate |
| `raw_oreb` | `oreb / oreb_chance` | Offensive rebound rate |
| `raw_otrans` | `trans / total_poss` | Transition rate |

**Defensive factors** (aggregated over possessions where player is on defense — what the opponent did).  
All are then **demeaned** (subtract league average) so the 4F regression intercept equals the overall average RAPM.

In [ ]:
def compute_factor_rates(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute per-player on-court four-factor rates from stints_rich.
    Returns one row per player_id with raw_ots, raw_otov, raw_oreb, raw_otrans
    and the defensive equivalents, plus poss_off / poss_def.
    """
    off_cols = [f"off_p{i}" for i in range(1, 6)]
    def_cols = [f"def_p{i}" for i in range(1, 6)]

    # Shared possession-level metrics
    df = df.copy()
    df["pts"]        = df["fgm"]*2 + df["fg3m"] + df["ftm"]
    df["shot_att"]   = df["fga"]*2 + df["fg3a"] + df["fta"]*0.44  # TS denominator
    df["poss_denom"] = df["fga"] + df["fta"]*0.44 + df["tov_flag"]

    off_agg = []
    def_agg = []

    for side, cols, agg_list in [("off", off_cols, off_agg), ("def", def_cols, def_agg)]:
        melted = (
            df[["pts","shot_att","poss_denom","tov_flag","oreb","oreb_chance","trans_flag"]
               + cols]
            .melt(id_vars=["pts","shot_att","poss_denom","tov_flag",
                           "oreb","oreb_chance","trans_flag"],
                  value_vars=cols, value_name="player_id")
            .dropna(subset=["player_id"])
        )
        melted["player_id"] = melted["player_id"].astype(int)

        g = (
            melted.groupby("player_id")
            .agg(
                poss       =("pts",         "count"),
                sum_pts    =("pts",         "sum"),
                sum_shot   =("shot_att",    "sum"),
                sum_poss_d =("poss_denom",  "sum"),
                sum_tov    =("tov_flag",    "sum"),
                sum_oreb   =("oreb",        "sum"),
                sum_oreb_c =("oreb_chance", "sum"),
                sum_trans  =("trans_flag",  "sum"),
            )
            .reset_index()
        )
        prefix = "o" if side == "off" else "d"
        g[f"raw_{prefix}ts"]    = g["sum_pts"]   / g["sum_shot"].clip(lower=1e-9)
        g[f"raw_{prefix}tov"]   = g["sum_tov"]   / g["sum_poss_d"].clip(lower=1e-9)
        g[f"raw_{prefix}reb"]   = g["sum_oreb"]  / g["sum_oreb_c"].clip(lower=1e-9)
        g[f"raw_{prefix}trans"] = g["sum_trans"] / g["poss"].clip(lower=1e-9)
        g = g.rename(columns={"poss": f"poss_{side}"})
        agg_list.append(
            g[["player_id", f"poss_{side}",
               f"raw_{prefix}ts", f"raw_{prefix}tov",
               f"raw_{prefix}reb", f"raw_{prefix}trans"]]
        )

    factors = off_agg[0].merge(def_agg[0], on="player_id", how="outer")
    return factors


print("Computing per-player four-factor rates...")
ff = compute_factor_rates(stints_rich)
print(f"Players with factor data: {len(ff):,}")

# League averages (weighted by possession count)
league_avg = {
    "ots":    (stints_rich["fgm"]*2 + stints_rich["fg3m"] + stints_rich["ftm"]).sum()
              / (stints_rich["fga"]*2 + stints_rich["fg3a"] + stints_rich["fta"]*0.44).clip(lower=1e-9).sum(),
    "otov":   stints_rich["tov_flag"].sum()
              / (stints_rich["fga"] + stints_rich["fta"]*0.44 + stints_rich["tov_flag"]).clip(lower=1e-9).sum(),
    "oreb":   stints_rich["oreb"].sum() / stints_rich["oreb_chance"].clip(lower=1e-9).sum(),
    "otrans": stints_rich["trans_flag"].sum() / len(stints_rich),
}
# Defensive league averages are the same pool (what opponents do on average)
league_avg.update({"dts": league_avg["ots"], "dtov": league_avg["otov"],
                   "dreb": league_avg["oreb"], "dtrans": league_avg["otrans"]})

print("\nLeague averages:")
for k, v in league_avg.items():
    print(f"  {k}: {v:.4f}")

# Demean
ff["ots"]    = ff["raw_ots"]    - league_avg["ots"]
ff["otov"]   = ff["raw_otov"]   - league_avg["otov"]
ff["oreb"]   = ff["raw_oreb"]   - league_avg["oreb"]
ff["otrans"] = ff["raw_otrans"] - league_avg["otrans"]
# For defense: positive dts = opponent TS% BELOW avg (good defense) → negate
ff["dts"]    = -(ff["raw_dts"]    - league_avg["dts"])
ff["dtov"]   =   ff["raw_dtov"]   - league_avg["dtov"]   # opponent TOV% above avg = good
ff["dreb"]   = -(ff["raw_dreb"]   - league_avg["dreb"])  # opponent OREB% below avg = good
ff["dtrans"] = -(ff["raw_dtrans"] - league_avg["dtrans"]) # opponent trans% below avg = good

FACTOR_COLS = ["ots", "otov", "oreb", "otrans", "dts", "dtov", "dreb", "dtrans"]
print("\nFactor data ready.")

## 4 · 4-Factor RAPM decomposition

OLS regression: `net_rapm ~ ots + otov + oreb + otrans + dts + dtov + dreb + dtrans`

The **betas** tell you how much each factor is worth in RAPM points.  
The **reconstructed RAPM** is the model's prediction — the **residual** is what the factors can't explain.

In [ ]:
from sklearn.linear_model import LinearRegression

# Merge RAPM with factor rates, keep only players with both
merged = rapm.merge(ff[["player_id"] + FACTOR_COLS], on="player_id", how="inner").dropna(subset=FACTOR_COLS)

X = merged[FACTOR_COLS].values
y = merged["net_rapm"].values

ols = LinearRegression(fit_intercept=True)
ols.fit(X, y)

merged["rapm_reconstructed"] = np.round(ols.predict(X), 3)
merged["residual"]           = np.round(merged["net_rapm"] - merged["rapm_reconstructed"], 3)

r2 = ols.score(X, y)
print(f"4F OLS fit — R² = {r2:.4f}  (factors explain {r2*100:.1f}% of RAPM variance)")
print(f"Intercept: {ols.intercept_:.4f}\n")

betas = pd.DataFrame({"factor": FACTOR_COLS, "beta": np.round(ols.coef_, 4)})
betas.style.format({"beta": "+.4f"}).background_gradient(subset=["beta"], cmap="RdYlGn").set_caption("4F RAPM betas")

## 5 · Final table

Combined RAPM + 4-factor decomposition, sorted by net RAPM.

In [ ]:
display_cols = [
    "player_name", "poss",
    "net_rapm", "orapm", "drapm",
    "rapm_reconstructed", "residual",
    "ots", "otov", "oreb", "otrans",
    "dts", "dtov", "dreb", "dtrans",
]

out = (
    merged[display_cols]
    .sort_values("net_rapm", ascending=False)
    .reset_index(drop=True)
)

rapm_fmt  = "+.2f"
fact_fmt  = "+.4f"
fmt_dict  = {
    "poss":               ",d",
    "net_rapm":           rapm_fmt,
    "orapm":              rapm_fmt,
    "drapm":              rapm_fmt,
    "rapm_reconstructed": rapm_fmt,
    "residual":           rapm_fmt,
    **{c: fact_fmt for c in FACTOR_COLS},
}

(
    out.head(40)
    .style
    .format(fmt_dict)
    .background_gradient(subset=["net_rapm", "orapm", "drapm"], cmap="RdYlGn")
    .background_gradient(subset=["ots", "oreb", "otrans"], cmap="RdYlGn")
    .background_gradient(subset=["otov"], cmap="RdYlGn_r")  # reversed: lower tov = better
    .background_gradient(subset=["dts", "dtov", "dreb", "dtrans"], cmap="RdYlGn")
    .set_caption(
        f"WNBA RAPM + 4F decomposition | "
        f"seasons: {', '.join(str(y) for y in RAPM_SEASONS)} | "
        f"λ={LAMBDA:,} | min {MIN_POSS} poss | 4F R²={r2:.3f}"
    )
)

In [ ]:
# Save to CSV
out_path = DATA_DIR / "rapm_and_4f_output.csv"
out.to_csv(out_path, index=False)
print(f"Saved → {out_path}")